# TGB: Time Gradient Boosting 
Application of ARIMA insprired TGB features (lags, rolling avgs, rolling std deviations) to convert classical ML classifiers into ones useful for time series classification. 

In theory this notebook will improve on LightGBM with Fixed Data.ipynb

## Load Data and Confirm Types

In [1]:
import pandas as pd

df = pd.read_parquet('../Data/EastAfrica Preprocessed/eastafrica.parquet')
print("Shape: ", df.shape)

Shape:  (2314906, 55)


In [2]:
for col in df.columns:
    print (f"{col}: ", df[col].dtype)

id:  category
name:  category
country:  category
latitude:  float64
longitude:  float64
minBorderDistanceKm:  float64
minCapitalDistanceKm:  float64
periodStart:  datetime64[ns]
LocalEventCount:  int64
LocalDistinctActorCount:  int64
LocalDistinctEventTypes:  int64
LocalDistinctEventSubtypes:  int64
LocalTotalFatalities:  int64
LocalMeanSeverity:  float64
LocalMaxSeverity:  int64
LocalProtestersEventCount:  int64
LocalStateForcesEventCount:  int64
LocalPoliticalMilitiaEventCount:  int64
LocalIdentityMilitiaEventCount:  int64
LocalRebelGroupEventCount:  int64
LocalRiotersEventCount:  int64
LocalCiviliansEventCount:  int64
LocalOtherTypeEventCount:  int64
RegionalEventCount:  int64
RegionalDistinctActorCount:  int64
RegionalDistinctEventTypes:  int64
RegionalDistinctEventSubTypes:  int64
RegionalTotalFatalities:  int64
RegionalMeanSeverity:  float64
RegionalMaxSeverity:  int64
RegionalProtestersEventCount:  int64
RegionalStateForcesEventCount:  int64
RegionalPoliticalMilitiaEventCount:  

## Application of Target Variables
* Targets will be "conflict at any point between t and t+n", not point predictions. This is to try and solve the sparse data issue.
* Targets will always take values from the future, not the past.
* There must be zero leakage between the series for this to work properly.
* Note that this sets the targets for the trailing periods (likely the last 12) to NaN, this is intended. Otherwise they would default to False, which is not accurate. 
    * That's OK. We will never train on them, only predict, so they are not required. 

In [3]:
#Not used, for reference only (or for reverting back to point predictions (code from LightGBM with Fixed Data.ipynb)
def generate_future_conflict_target_point(df, horizons, target_columns = []):
    # Ensure correct ordering inside each series
    df = df.sort_values(["id", "periodStart"])

    # Columns that define conflict
    local_cols = ["LocalTotalFatalities", "LocalAvgSeverity"]
    regional_cols = ["RegionalTotalFatalities", "RegionalAvgSeverity"]

    for h in horizons:
        # Shift forward into the FUTURE (t + h)
        shifted_local = df.groupby("id", observed = True)[local_cols].shift(-h)
        shifted_regional = df.groupby("id", observed = True)[regional_cols].shift(-h)

        # Binary conflict flags
        df[f"TargetConflictLocal_t+{h}"] = (shifted_local >= 3).any(axis=1)
        target_columns.append(f"TargetConflictLocal_t+{h}")
        #df = df[f"TargetConflictLocal_t+{h}"].astype('category')
        
        
        df[f"TargetConflictRegional_t+{h}"] = (shifted_regional >= 3).any(axis=1)
        target_columns.append(f"TargetConflictRegional_t+{h}")
       # df = df[f"TargetConflictLocal_t+{h}"].astype('category')
        
    return df

In [4]:
def generate_future_conflict_targets_interval(df, horizons, target_columns=None):
    if target_columns is None:
        target_columns = []

    df = df.sort_values(["id", "periodStart"])

    local_cols = ["LocalTotalFatalities", "LocalMeanSeverity"]
    regional_cols = ["RegionalTotalFatalities", "RegionalMeanSeverity"]

    for h in horizons:

        # Rolling window into the future
        local_window = (
            df.groupby("id", observed = False)[local_cols]
              .rolling(window=h, min_periods=1)
              .max()
              .shift(-h+1)
              .reset_index(level=0, drop=True)
        )

        regional_window = (
            df.groupby("id", observed = False)[regional_cols]
              .rolling(window=h, min_periods=1)
              .max()
              .shift(-h+1)
              .reset_index(level=0, drop=True)
        )

        # Identify rows where the window is entirely NaN (no future data)
        no_future_local = local_window.isna().all(axis=1)
        no_future_regional = regional_window.isna().all(axis=1)

        # Conflict occurs at ANY point in the interval
        local_target = (local_window >= 3).any(axis=1).astype("boolean")
        regional_target = (regional_window >= 3).any(axis=1).astype("boolean")

        # Replace rows with no future data with NaN
        local_target[no_future_local] = pd.NA
        regional_target[no_future_regional] = pd.NA

        df[f"TargetConflictLocal_t+{h}"] = local_target
        df[f"TargetConflictRegional_t+{h}"] = regional_target

        target_columns.extend([
            f"TargetConflictLocal_t+{h}",
            f"TargetConflictRegional_t+{h}"
        ])

    return df

In [5]:
import gc
target_columns = []
startTime = pd.Timestamp.now()
horizons = [1, 3, 6, 12]
df = generate_future_conflict_targets_interval(df, horizons = horizons, target_columns = target_columns) #will convert to nullable boolean type

gc.collect()
print("generate_future_conflict_targets_interval: ", pd.Timestamp.now() - startTime)

generate_future_conflict_targets_interval:  0 days 00:00:15.584477


### Class Breakdown Over Entire Dataset

In [6]:
for col in target_columns:
    counts = df[col].value_counts(dropna=False)
    total = counts.sum()
    
    positives = counts.get(True, 0)
    negatives = counts.get(False, 0)
    
    print(f"\nColumn: {col}")
    print(f"  Positives: {positives} ({positives/total:.4f})")
    print(f"  Negatives: {negatives} ({negatives/total:.4f})")

    


Column: TargetConflictLocal_t+1
  Positives: 152423 (0.0658)
  Negatives: 2162483 (0.9342)

Column: TargetConflictRegional_t+1
  Positives: 1457846 (0.6298)
  Negatives: 857060 (0.3702)

Column: TargetConflictLocal_t+3
  Positives: 295728 (0.1277)
  Negatives: 2019176 (0.8722)

Column: TargetConflictRegional_t+3
  Positives: 1859153 (0.8031)
  Negatives: 455751 (0.1969)

Column: TargetConflictLocal_t+6
  Positives: 431288 (0.1863)
  Negatives: 1883613 (0.8137)

Column: TargetConflictRegional_t+6
  Positives: 2034426 (0.8788)
  Negatives: 280475 (0.1212)

Column: TargetConflictLocal_t+12
  Positives: 600305 (0.2593)
  Negatives: 1714590 (0.7407)

Column: TargetConflictRegional_t+12
  Positives: 2141819 (0.9252)
  Negatives: 173076 (0.0748)


### Conflict and Peace Prone Places (Whole Dataset)
Ignoring places where conflict predictions are 100% conflict or 100% peace

In [7]:
results = {}

for col in target_columns:
    # Compute conflict rate per id
    id_rates = df.groupby("id", observed = False)[col].mean()

    # Exclude trivial cases
    filtered = id_rates[(id_rates > 0) & (id_rates < 1)]

    # Sort
    most_conflict_prone = filtered.sort_values(ascending=False).head(10)
    most_peaceful = filtered.sort_values(ascending=True).head(10)

    # Store results
    results[col] = {
        "most_conflict_prone": most_conflict_prone,
        "most_peaceful": most_peaceful
    }

    # Print nicely
    print(f"\n=== {col} ===")
    print("Most conflict-prone IDs (excluding 100% conflict):")
    print(most_conflict_prone)

    print("\nMost peaceful IDs (excluding 0% conflict):")
    print(most_peaceful)



=== TargetConflictLocal_t+1 ===
Most conflict-prone IDs (excluding 100% conflict):
id
Somalia_Mogadishu_-_Kaxda    0.992806
Somalia_Dhagahtur            0.992806
Somalia_Siinka_Dheer         0.992806
Somalia_K13                  0.992806
Somalia_Mogadishu_-_Karan    0.992806
Somalia_Lafoole              0.992806
Somalia_Xaafadda_Taree       0.992806
Somalia_Xaawo-Cabdi          0.992806
Somalia_Buulo-Folyo          0.992806
Somalia_Buufow_Bacaad        0.992806
Name: TargetConflictLocal_t+1, dtype: Float64

Most peaceful IDs (excluding 0% conflict):
id
Sudan_Al_Fadniya_Al_Miregat    0.007194
South_Sudan_Lohila             0.007194
Ethiopia_Menze_Lalo_Midir      0.007194
South_Sudan_Lith               0.007194
South_Sudan_Liet               0.007194
Ethiopia_Mergech               0.007194
South_Sudan_Langkei            0.007194
South_Sudan_Kwaich             0.007194
South_Sudan_Kursomba           0.007194
South_Sudan_Kulerau            0.007194
Name: TargetConflictLocal_t+1, dtype: Fl

## Apply TGB Features

### Possible Kernal Failure!
The kernal likes to fall over on these chunks. For best chance of it actually completing:
1. Run all cells up to this one.
2. Then run the column application chunks one at a time.
3. Then run all following cells. 

In [8]:
acled_raw_features = ['LocalEventCount',
'LocalDistinctActorCount',
'LocalDistinctEventTypes',
'LocalDistinctEventSubtypes',
'LocalTotalFatalities',
'LocalMeanSeverity',
'LocalMaxSeverity',
'LocalProtestersEventCount',
'LocalStateForcesEventCount',
'LocalPoliticalMilitiaEventCount',
'LocalIdentityMilitiaEventCount',
'LocalRebelGroupEventCount',
'LocalRiotersEventCount',
'LocalCiviliansEventCount',
'LocalOtherTypeEventCount',
'RegionalEventCount',
'RegionalDistinctActorCount',
'RegionalDistinctEventTypes',
'RegionalDistinctEventSubTypes',
'RegionalTotalFatalities',
'RegionalMeanSeverity',
'RegionalMaxSeverity',
'RegionalProtestersEventCount',
'RegionalStateForcesEventCount',
'RegionalPoliticalMilitiaEventCount',
'RegionalIdentityMilitiaEventCount',
'RegionalCiviliansEventCount',
'RegionalRebelGroupEventCount',
'RegionalRiotersEventCount',
'RegionalOtherTypeEventCount',]

country_facts_scaled_features =['Control_of_Corruption__Governance_estimate_approx_25_to_+25_GOV_WGI_CC_EST',
'Rule_of_Law__Governance_estimate_approx_25_to_+25_GOV_WGI_RL_EST',
'Primary_completion_rate_total__of_relevant_age_group_SEPRMCMPTZS',
'Children_out_of_school__of_primary_school_age_SEPRMUNERZS',
'GNI_per_capita_constant_2015_US_NYGNPPCAPKD',
'GDP_per_capita_constant_2015_US_NYGDPPCAPKD',
'Armed_forces_personnel__of_total_labor_force_MSMILTOTLTFZS',
'Military_expenditure__of_GDP_MSMILXPNDGDZS',
'Renewable_internal_freshwater_resources_per_capita_cubic_meters_ERH2OINTRPC',
'Cereal_yield_kg_per_hectare_AGYLDCRELKG',
'Permanent_cropland__of_land_area_AGLNDCROPZS',
'Water_productivity_total_constant_2015_US_GDP_per_cubic_meter_of_total_freshwater_withdrawal_ERGDPFWTLM3KD',]

country_facts_unscaled_features = ['Population_total_SPPOPTOTL',
                                   'Internally_displaced_persons_new_displacement_associated_with_conflict_and_violence_number_of_cases_VCIDPNWCV']

tgb_lags = [1, 2, 3, 6, 12, 18]
tgb_windows = [ 2, 3, 6, 12, 18]

In [9]:
import gc

#raw values from the previous n period. 
def generate_lag_features(df, feature_cols, lags):
    # Ensure correct ordering inside each series
  #  df = df.sort_values(["id", "periodStart"])

    for col in feature_cols:
        for h in lags:
            df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)

    gc.collect()
    return df

#Rolling median over previous n periods. 
def generate_rolling_median_features(df, feature_cols, windows):
    # Ensure correct ordering inside each series
  #  df = df.sort_values(["id", "periodStart"])

    for col in feature_cols:
        for w in windows:
            df[f"{col}_median{w}"] = (
                df.groupby("id",  observed = True)[col]
                  .rolling(window=w, min_periods=1)
                  .median()
                  .reset_index(level=0, drop=True)
            )
    gc.collect()
    return df

def generate_rolling_mean_features(df, feature_cols, windows):
    # Ensure correct ordering inside each series
  #  df = df.sort_values(["id", "periodStart"])

    for col in feature_cols:
        for w in windows:
            df[f"{col}_mean{w}"] = (
                df.groupby("id",  observed = True)[col]
                  .rolling(window=w, min_periods=1)
                  .mean()
                  .reset_index(level=0, drop=True)
            )
    gc.collect()
    return df

#rolling std deviation (volatility over previous n periods
def generate_rolling_std_features(df, feature_cols, windows):
    # Ensure correct ordering inside each series
   # df = df.sort_values(["id", "periodStart"])

    for col in feature_cols:
        for w in windows:
            df[f"{col}_std{w}"] = (
                df.groupby("id",  observed = True)[col]
                  .rolling(window=w, min_periods=1)
                  .std()
                  .reset_index(level=0, drop=True)
            )
    gc.collect()
    return df

### Rolling Std Deviations (Volatility)
09:43 minutes to complete

In [10]:
startTime = pd.Timestamp.now()
df = generate_rolling_std_features(df, acled_raw_features, tgb_windows) # 3 mins
print("generate_rolling_std_features: ", pd.Timestamp.now() - startTime)

C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using 

generate_rolling_std_features:  0 days 00:03:24.733802


C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std{w}"] = (


In [11]:
#convert float64s to float32 to save memory:
float_cols = df.select_dtypes(include=["float64"]).columns

df[float_cols] = df[float_cols].astype("float32")

### Rolling Lags
Very quick to complete: in the order of seconds. 

In [12]:
#Initial sort: prevents being called n times (removed from the column generation functions)
df = df.sort_values(["id", "periodStart"])

In [13]:
startTime = pd.Timestamp.now()
df = generate_lag_features(df, acled_raw_features, tgb_lags) #9 mins
print("generate_lag_features: ", pd.Timestamp.now() - startTime)

C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.

generate_lag_features:  0 days 00:00:09.742047


In [14]:
#convert float64s to float32 to save memory:
float_cols = df.select_dtypes(include=["float64"]).columns

df[float_cols] = df[float_cols].astype("float32")

### Rolling Medians
Medians are less susceptible to extreme outliers, but this might not be desirable for this purpose. To remove if the models do not rank them highly.
-15 mins

### Rolling Means

In [15]:
startTime = pd.Timestamp.now()
df = generate_rolling_mean_features(df, acled_raw_features, tgb_windows) #5 mins
print("generate_rolling_mean_features: ", pd.Timestamp.now() - startTime)

C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_mean{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_mean{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

generate_rolling_mean_features:  0 days 00:03:19.752422


C:\Users\andre\AppData\Local\Temp\ipykernel_45672\4149307426.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_mean{w}"] = (


In [16]:
#convert float64s to float32 to save memory:
float_cols = df.select_dtypes(include=["float64"]).columns

df[float_cols] = df[float_cols].astype("float32")

## Generate Scaled Country Fact Features

In [17]:
import numpy as np
def generate_unscaled_lag_features(df, features, lags):
   #Generate log 
    for feature in features:
        for lag in lags:

            #temporary, it's just the raw values. 
            lag_col = f"{feature}_t{lag}"
            log_col = f"{feature}_t{lag}_log" #rate of change. 
            
            #create lagged feature
            df[lag_col] = df.groupby("country")[feature].shift(lag)
            
            # Log transform the lagged feature to capture the rate of change:
            df[log_col] = np.log1p(df[lag_col])

            #delete the temporary column:
            df.drop(columns=[lag_col], inplace=True)
    
    return df

def generate_scaled_rate_features(df, features, lag_pair):
    #Applies rate of change columns as a percentage. Intended for the previous 13 and 26 lags. 

    lag1, lag2 = lag_pair

    index = 0
    for feature in features:
        index = index + 1
        # Temporary lag columns
        lag1_col = f"{feature}_t{lag1}"
        lag2_col = f"{feature}_t{lag2}"

        #create temportary lagged features
        df[lag1_col] = df.groupby("country")[feature].shift(lag1)
        df[lag2_col] = df.groupby("country")[feature].shift(lag2)

        #rate of change between t-lag1 and t-lag2
        rate_col = f"{feature}_rate_t{lag1}_t{lag2}"
        df[rate_col] = (df[lag1_col] - df[lag2_col]) / df[lag2_col]

        #delete temporary lag columns
        df.drop(columns=[lag1_col, lag2_col], inplace=True)

        print(f"done feature {index} of {len(features)}")
    return df

### Apply Rate-of-Change to Scaled Country Fact Features
Windows of 13 and 26 are used to get the rate of change from the second previous year compared to the previous year, as it's assumed values for the current year will not be available (and likely never will be, to avoid the World Bank needing to release for incomplete years).  Again, these chunks like to fail on memory usage so the same instructions apply. 

In [18]:
gc.collect()
df = df.sort_values(["country", "periodStart"])

#Apply rate of change to the scaled features:
startTime = pd.Timestamp.now()
df = generate_scaled_rate_features(df, country_facts_scaled_features, [13, 26]) #1 year, 2 years. 
print("generate_scaled_rate_features: ", pd.Timestamp.now() - startTime) #takes ~5 mins. If it runs: likes to fail on RAM usage. 

C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavio

done feature 1 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavio

done feature 2 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavio

...

done feature 10 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavio

done feature 11 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavio

done feature 12 of 12
generate_scaled_rate_features:  0 days 00:00:56.761444


### Scale and apply Rate-of-Change to Unscaled Country Fact Features
-6 mins

In [19]:
startTime = pd.Timestamp.now()
df = generate_unscaled_lag_features(df, country_facts_unscaled_features, [13, 26]) #1 year, 2 years. 
print("generate_unscaled_lag_features: ", pd.Timestamp.now() - startTime) #47s 

C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag_col] = df.groupby("country")[feature].shift(lag)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag_col] = df.groupby("country")[feature].shift(lag)
C:\Users\andre\AppData\Local\Temp\ipykernel_45672\3924688775.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

generate_unscaled_lag_features:  0 days 00:00:18.293441


### Remove Raw features

In [20]:
gc.collect()
df = df[df.columns.drop(acled_raw_features)] #drop the raw features, not required now.
df = df[df.columns.drop(country_facts_scaled_features)] #drop the raw features, not required now.
df = df[df.columns.drop(country_facts_unscaled_features)] #drop the raw features, not required now.

## Check Column Types

In [21]:
for col in df.columns:
    print (f"{col}: ", df[col].dtype)
print("Shape: ", df.shape)

id:  category
name:  category
country:  category
latitude:  float32
longitude:  float32
minBorderDistanceKm:  float32
minCapitalDistanceKm:  float32
periodStart:  datetime64[ns]
month_sin:  float32
month_cos:  float32
is_voting:  bool
TargetConflictLocal_t+1:  boolean
TargetConflictRegional_t+1:  boolean
TargetConflictLocal_t+3:  boolean
TargetConflictRegional_t+3:  boolean
TargetConflictLocal_t+6:  boolean
TargetConflictRegional_t+6:  boolean
TargetConflictLocal_t+12:  boolean
TargetConflictRegional_t+12:  boolean
LocalEventCount_std2:  float32
LocalEventCount_std3:  float32
LocalEventCount_std6:  float32
LocalEventCount_std12:  float32
LocalEventCount_std18:  float32
LocalDistinctActorCount_std2:  float32
LocalDistinctActorCount_std3:  float32
LocalDistinctActorCount_std6:  float32
LocalDistinctActorCount_std12:  float32
LocalDistinctActorCount_std18:  float32
LocalDistinctEventTypes_std2:  float32
LocalDistinctEventTypes_std3:  float32
LocalDistinctEventTypes_std6:  float32
LocalDis

## Drop the Median Columns: suspected data leakage. 

In [22]:
#Drop columns named median: suspect leakage.
df = df.drop(columns=df.filter(regex="Median").columns)
df = df.drop(columns=df.filter(regex="median").columns)

## Convert Float64 to float32 for memory issues

In [23]:
#convert float64s to float32 to save memory:
float_cols = df.select_dtypes(include=["float64"]).columns

df[float_cols] = df[float_cols].astype("float32")

## Drop the Max columns: Suspect leakage. 

In [24]:
#Drop columns named median: suspect leakage.
df = df.drop(columns=df.filter(regex="Max").columns)

## Save the TGB-Ready Dataset
750Mb!

In [25]:
df = df.sort_values(by=["periodStart", "id"], ascending=True) #sort by periodStart, then id.
df.to_parquet('../Data/EastAfrica Preprocessed/eastafrica-ready-for-tgb.parquet')
print("Rows x columns: ", df.shape)

Rows x columns:  (2314906, 483)


# Future Actions
* Train/Test Split (by periodStart)
    * Export the test and training datasets to Parquet files.  
* Model training: CatBoost, LightGBM, RF
  * Model saving as Pickle files
* Ensemble classifier- one per target, with three models each. 